# Answers 03 Grid matrices

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How is a net of 6 × 6 cells written as matrices?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

Nodes $n_0 \ldots n_{48}$ are numbered row by row from the bottom left, 1 m apart. Bars $e_0 \ldots e_{41}$ go to the right neighbour, bars $e_{42} \ldots e_{83}$ go to the upper neighbour.

Write down:

- the branch node matrix $C$
- the vector of force densities $q$
- the vectors of coordinates $x$, $y$, $z$

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">


# Network

In [1]:
import numpy as np

# nodes: 6 x 6 cells = 7 x 7 nodes n0..n48, numbered row by row from the bottom left
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

# bars: each node to its right and upper neighbour
edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]

print("nodes:", n * n)
print("bars: ", len(edges))

nodes: 49
bars:  84


# Connectivity Matrix

One row per bar, one column per node. In the row of a bar: $1$ at its first node, $-1$ at its second node, $0$ everywhere else. The first two rows:

$$
\newcommand{\cell}[1]{\hphantom{-}\mathclap{#1}\hphantom{-}}
\begin{array}{rcc}
 & & \begin{matrix} \cell{n_0} & \cell{n_1} & \cell{n_2} & \cell{n_3} & \cell{\cdots} & \cell{n_{48}} \end{matrix} \\[2pt]
C = & \begin{matrix} e_0 \\ e_1 \\ \vdots \end{matrix} &
\left[ \;
\begin{matrix}
\cell{1} & \cell{-1} & \cell{0} & \cell{0} & \cell{\cdots} & \cell{0} \\
\cell{0} & \cell{1} & \cell{-1} & \cell{0} & \cell{\cdots} & \cell{0} \\
\cell{\vdots} & \cell{} & \cell{} & \cell{} & \cell{} & \cell{\vdots}
\end{matrix}
\; \right]
\end{array}
$$

In [2]:
# connectivity matrix
C = np.zeros((len(edges), n*n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

print("C has", C.shape[0], "rows (bars) and", C.shape[1], "columns (nodes)")
print("bar e0 connects nodes", edges[0])
print("row e0 of C:", C[0].astype(int))

C has 84 rows (bars) and 49 columns (nodes)
bar e0 connects nodes (0, 1)
row e0 of C: [ 1 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0]


# Viewer

The net, and the matrix $C$ beside it. In the matrix every row is a bar and every column is a node.

In [3]:
from session_py import Color, Matrix, Session, Xform
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, loads green
RED   = Color(0.84, 0.16, 0.16, 1.0)
BLUE  = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY  = Color(0.70, 0.70, 0.70, 1.0)

session = Session("fdm_exercise_3")

session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()))

# the matrix C, to the right of the net
matrix = Matrix.from_rows(C.tolist())
matrix.name = "C"
matrix.cell = 0.12                 # size of one square of the matrix
session.add_matrix(matrix)
session.set_xform(matrix.guid, Xform.translation(8.0, 8.0, 0.0))

show(session, "fdm_exercise_3", view="top", height=700)

# Force Densities

One number per bar. Start with 1 everywhere:

$$ q = [1, 1, \ldots, 1] \text{ kN/m} $$

In [4]:
# force densities, one per bar
q = np.ones(len(edges))
print(q)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


# Coordinates

One number per node. The net starts flat, so $z = 0$.

In [5]:
print("x =", x)
print("y =", y)
print("z =", z)

x = [0. 1. 2. 3. 4. 5. 6. 0. 1. 2. 3. 4. 5. 6. 0. 1. 2. 3. 4. 5. 6. 0. 1. 2.
 3. 4. 5. 6. 0. 1. 2. 3. 4. 5. 6. 0. 1. 2. 3. 4. 5. 6. 0. 1. 2. 3. 4. 5.
 6.]
y = [0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 2. 2. 2. 2. 2. 2. 2. 3. 3. 3.
 3. 3. 3. 3. 4. 4. 4. 4. 4. 4. 4. 5. 5. 5. 5. 5. 5. 5. 6. 6. 6. 6. 6. 6.
 6.]
z = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0.]


# Check

The coordinate differences are found by multiplying the connectivity matrix with the node coordinates, so the bar lengths come straight from $C$:

$$ u = C\,x \qquad v = C\,y \qquad w = C\,z \qquad l = \sqrt{u^2 + v^2 + w^2} $$

Every bar of the grid is 1 m long.

In [6]:
l = np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)
print(l)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
